# SimpleVLA Training Notebook
**Vizuara AI Labs | VLA Driving Simulator**

Train a Vision-Language-Action model on driving data collected from the browser simulator.

### How to use
1. Collect driving data in the simulator (Training Mode), download the JSON file(s)
2. Upload your `vla_dataset_*.json` file(s) below
3. Run all cells to train the model (~2 min on a free Colab GPU)
4. Download `model.json` and load it in the simulator's Inference Mode

## 1. Upload Dataset Files
Upload one or more `vla_dataset_*.json` files you downloaded from the simulator.

In [ ]:
from google.colab import files
import json
import os

print("Select your vla_dataset_*.json files:")
uploaded = files.upload()

# Merge all uploaded datasets, dedup by timestamp
all_samples = []
seen_timestamps = set()
all_intent_labels = None

for filename, content in sorted(uploaded.items()):
    if not filename.endswith('.json'):
        print(f"  Skipping {filename} (not JSON)")
        continue
    d = json.loads(content)
    meta = d.get('metadata', {})
    samples = d.get('samples', [])

    # Track intent labels from the first valid file
    if all_intent_labels is None and 'intent_labels' in meta:
        all_intent_labels = meta['intent_labels']

    count = 0
    for s in samples:
        ts = s.get('timestamp', id(s))
        if ts not in seen_timestamps:
            all_samples.append(s)
            seen_timestamps.add(ts)
            count += 1
    print(f"  {filename}: {count} new samples (total so far: {len(all_samples)})")

if all_intent_labels is None:
    all_intent_labels = ['drive']

print(f"\nTotal samples: {len(all_samples)}")
print(f"Intent labels: {all_intent_labels}")

## 2. Analyze Dataset

In [ ]:
from collections import Counter

# Auto-detect number of intents from the data
unique_intents = sorted(set(s['language_id'] for s in all_samples))
num_intents = max(unique_intents) + 1
print(f"Detected {num_intents} intent(s): {unique_intents}")

# Extend labels if needed
while len(all_intent_labels) < num_intents:
    all_intent_labels.append(f'intent_{len(all_intent_labels)}')

# Intent distribution
lang_dist = Counter(s['language_id'] for s in all_samples)
print("\nIntent distribution:")
for lid in sorted(lang_dist):
    label = all_intent_labels[lid] if lid < len(all_intent_labels) else f'intent_{lid}'
    print(f"  [{lid}] {label}: {lang_dist[lid]} samples")

# Action distribution
n = len(all_samples)
fwd = sum(1 for s in all_samples if s['actions']['forward'])
bwd = sum(1 for s in all_samples if s['actions']['backward'])
lft = sum(1 for s in all_samples if s['actions']['left'])
rgt = sum(1 for s in all_samples if s['actions']['right'])

print(f"\nAction positive rates:")
print(f"  forward:  {fwd:>5}/{n} ({fwd/n:>5.1%})")
print(f"  backward: {bwd:>5}/{n} ({bwd/n:>5.1%})")
print(f"  left:     {lft:>5}/{n} ({lft/n:>5.1%})")
print(f"  right:    {rgt:>5}/{n} ({rgt/n:>5.1%})")

## 3. Prepare Data
Decode base64 images into tensors and create train/val splits.

In [ ]:
import base64
import io
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

IMG_SIZE = 64

# Fix random seeds for reproducible training
np.random.seed(42)
torch.manual_seed(42)
np.random.shuffle(all_samples)

# Pre-decode all images
t0 = time.time()
n = len(all_samples)
images = torch.zeros(n, 3, IMG_SIZE, IMG_SIZE)
langs = torch.zeros(n, dtype=torch.long)
actions = torch.zeros(n, 4)

for i, s in enumerate(all_samples):
    img_b64 = s['image'].split(',')[1] if ',' in s['image'] else s['image']
    img = Image.open(io.BytesIO(base64.b64decode(img_b64))).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    images[i] = torch.from_numpy(np.array(img, dtype=np.float32).transpose(2, 0, 1) / 255.0)
    langs[i] = s['language_id']
    actions[i] = torch.tensor([
        s['actions']['forward'],
        s['actions']['backward'],
        s['actions']['left'],
        s['actions']['right'],
    ], dtype=torch.float32)

    if (i + 1) % 500 == 0:
        print(f"  Decoded {i+1}/{n}...", end='\r')

print(f"Decoded {n} images in {time.time()-t0:.1f}s")

# Compute pos_weight to handle class imbalance
# (turning actions are rarer than forward, so upweight them)
pos_counts = actions.sum(dim=0)
neg_counts = n - pos_counts
pos_weights = torch.ones(4)
for j in range(4):
    if pos_counts[j] > 0:
        pos_weights[j] = min(neg_counts[j] / pos_counts[j], 10.0)

print(f"pos_weight: fwd={pos_weights[0]:.1f}  bwd={pos_weights[1]:.1f}  left={pos_weights[2]:.1f}  right={pos_weights[3]:.1f}")

# Train/val split (85/15)
split = int(n * 0.85)
print(f"Train: {split}, Val: {n - split}")

## 4. Dataset with Augmentation

In [ ]:
class VLADataset(Dataset):
    """VLA dataset with optional image augmentation."""

    def __init__(self, images, langs, actions, augment=False):
        self.images = images
        self.langs = langs
        self.actions = actions
        self.augment = augment

    def __len__(self):
        return self.images.shape[0]

    def __getitem__(self, idx):
        img = self.images[idx]
        lang = self.langs[idx]
        act = self.actions[idx]

        if self.augment:
            # Random horizontal flip -- only safe with 1 intent (e.g. curved track).
            # With multiple intents (e.g. fork road), flipping swaps the visual
            # path but keeps the same intent label, which confuses the model.
            if num_intents == 1 and torch.rand(1).item() < 0.5:
                img = img.flip(2)  # flip width
                act = act.clone()
                act[2], act[3] = act[3].item(), act[2].item()

            # Random brightness jitter (+/- 20%)
            img = img * (0.8 + torch.rand(1).item() * 0.4)

            # Random contrast jitter (+/- 20%)
            mean = img.mean()
            factor = 0.8 + torch.rand(1).item() * 0.4
            img = (img - mean) * factor + mean

            # Small Gaussian noise
            img = img + torch.randn_like(img) * 0.02
            img = img.clamp(0.0, 1.0)

        return img, lang, act

train_ds = VLADataset(images[:split], langs[:split], actions[:split], augment=True)
val_ds   = VLADataset(images[split:], langs[split:], actions[split:])

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=64)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## 5. Define Model

**SimpleVLA architecture:**
- Vision encoder: 3-layer CNN (3 -> 16 -> 32 -> 64) + global avg pool -> 64-dim
- Language encoder: Embedding(num_intents, 32) -> 32-dim
- Action decoder: Linear(96 -> 64) -> ReLU -> Linear(64 -> 4)

In [ ]:
class SimpleVLA(nn.Module):
    def __init__(self, num_intents=1, lang_dim=32):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, stride=2, padding=1)   # -> [16, 32, 32]
        self.conv2 = nn.Conv2d(16, 32, 3, stride=2, padding=1)  # -> [32, 16, 16]
        self.conv3 = nn.Conv2d(32, 64, 3, stride=2, padding=1)  # -> [64, 8, 8]
        self.pool = nn.AdaptiveAvgPool2d(1)                      # -> [64, 1, 1]
        self.lang_embed = nn.Embedding(num_intents, lang_dim)
        self.fc1 = nn.Linear(64 + lang_dim, 64)
        self.fc2 = nn.Linear(64, 4)
        self.relu = nn.ReLU()

    def forward(self, img, lang):
        x = self.relu(self.conv1(img))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.pool(x).flatten(1)          # [B, 64]
        l = self.lang_embed(lang)             # [B, 32]
        z = self.relu(self.fc1(torch.cat([x, l], 1)))  # [B, 64]
        return self.fc2(z)                    # [B, 4]

LANG_DIM = 32
model = SimpleVLA(num_intents=num_intents, lang_dim=LANG_DIM).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model: SimpleVLA ({num_params:,} parameters)")
print(f"  num_intents = {num_intents}")
print(f"  lang_dim    = {LANG_DIM}")
print(model)

## 6. Train

In [ ]:
EPOCHS = 100
LR = 0.001

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    for imgs, lngs, acts in train_dl:
        imgs, lngs, acts = imgs.to(device), lngs.to(device), acts.to(device)
        logits = model(imgs, lngs)
        loss = criterion(logits, acts)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * imgs.size(0)
    scheduler.step()
    epoch_loss /= len(train_ds)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, lngs, acts in val_dl:
            imgs, lngs, acts = imgs.to(device), lngs.to(device), acts.to(device)
            logits = model(imgs, lngs)
            val_loss += criterion(logits, acts).item() * imgs.size(0)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == acts).sum().item()
            total += acts.numel()
    val_loss /= len(val_ds)
    val_acc = correct / total

    history['train_loss'].append(epoch_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.time() - t0
        print(f"Epoch {epoch:3d}/{EPOCHS}  "
              f"train_loss={epoch_loss:.4f}  "
              f"val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.3f}  "
              f"[{elapsed:.0f}s]")

    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.0f}s")
print(f"Best validation accuracy: {best_val_acc:.3f}")

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history['val_acc'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].axhline(y=best_val_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.3f}')
axes[1].legend()

# Per-action accuracy on validation set
model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()
all_preds = []
all_acts = []
with torch.no_grad():
    for imgs, lngs, acts in val_dl:
        imgs, lngs = imgs.to(device), lngs.to(device)
        preds = (torch.sigmoid(model(imgs, lngs)) > 0.5).float().cpu()
        all_preds.append(preds)
        all_acts.append(acts)
all_preds = torch.cat(all_preds)
all_acts = torch.cat(all_acts)

action_names = ['forward', 'backward', 'left', 'right']
per_action_acc = [(all_preds[:, i] == all_acts[:, i]).float().mean().item() for i in range(4)]
colors = ['#4CAF50', '#FF9800', '#2196F3', '#E91E63']
axes[2].bar(action_names, per_action_acc, color=colors)
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Per-Action Accuracy (Val)')
axes[2].set_ylim(0, 1.05)
for i, v in enumerate(per_action_acc):
    axes[2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## 8. Evaluate Per-Intent Predictions

In [ ]:
ACTION_NAMES = ['fwd', 'bwd', 'left', 'right']

for lid in range(num_intents):
    label = all_intent_labels[lid] if lid < len(all_intent_labels) else f'intent_{lid}'
    mask = langs == lid
    subset_imgs = images[mask][:200].to(device)
    subset_langs = langs[mask][:200].to(device)

    if len(subset_imgs) == 0:
        print(f"[{lid}] {label}: no samples")
        continue

    with torch.no_grad():
        preds = (torch.sigmoid(model(subset_imgs, subset_langs)) > 0.5).int().cpu()

    pred_counts = Counter()
    for p in preds:
        keys = [ACTION_NAMES[i] for i in range(4) if p[i]]
        combo = '+'.join(keys) if keys else 'none'
        pred_counts[combo] += 1

    # Sort by count descending
    sorted_preds = sorted(pred_counts.items(), key=lambda x: -x[1])
    print(f"\n[{lid}] {label} ({mask.sum().item()} total samples, showing 200):")
    for combo, count in sorted_preds:
        print(f"  {combo:>20s}: {count:>4d} ({count/len(preds):>5.1%})")

## 9. Export model.json for Browser Inference
Export weights in the format the simulator's JS inference engine expects.

In [ ]:
model.load_state_dict(torch.load('best_model.pt', weights_only=True))
state = model.cpu().state_dict()

weights = {
    'conv1_weight': state['conv1.weight'].flatten().tolist(),
    'conv1_bias':   state['conv1.bias'].tolist(),
    'conv2_weight': state['conv2.weight'].flatten().tolist(),
    'conv2_bias':   state['conv2.bias'].tolist(),
    'conv3_weight': state['conv3.weight'].flatten().tolist(),
    'conv3_bias':   state['conv3.bias'].tolist(),
    'lang_embedding': state['lang_embed.weight'].flatten().tolist(),
    'fc1_weight':   state['fc1.weight'].flatten().tolist(),
    'fc1_bias':     state['fc1.bias'].tolist(),
    'fc2_weight':   state['fc2.weight'].flatten().tolist(),
    'fc2_bias':     state['fc2.bias'].tolist(),
}

export_data = {
    'format': 'simple_vla_js',
    'num_params': num_params,
    'num_intents': num_intents,
    'lang_dim': LANG_DIM,
    'intent_labels': all_intent_labels[:num_intents],
    'weights': weights,
}

with open('model.json', 'w') as f:
    json.dump(export_data, f)

size_kb = os.path.getsize('model.json') / 1024
print(f"Exported model.json ({size_kb:.0f} KB, {num_params:,} params)")
print(f"  format:      {export_data['format']}")
print(f"  num_intents: {export_data['num_intents']}")
print(f"  lang_dim:    {export_data['lang_dim']}")
print(f"  labels:      {export_data['intent_labels']}")

## 10. Download model.json
Load this file in the simulator: **Inference Mode -> Load Model -> select model.json -> Start Autopilot**

In [ ]:
files.download('model.json')
print("Done! Load model.json in the VLA Driving Simulator.")